In [ ]:
import os
import subprocess
import sys
from IPython.display import Markdown, display

#@markdown ## 1) 準備
#@markdown このセルでは、処理に必要な環境を整えます。
#@markdown 左の ▶ ボタンを押して、そのまま終わるまでお待ちください。

INPUT_DIR = "/content/処理待ち"
OUTPUT_DIR = "/content/処理済み"
MODEL_DIR = "/content/_pas_models"
TEMP_DIR = "/content/_processing_tmp"
REPO_DIR = "/content/python-audio-separator"
REPO_URL = "https://github.com/nomadkaraoke/python-audio-separator.git"
REPO_COMMIT = "0a644dbb07d94a7e747d5265e3abbaf600459449"

for path in [INPUT_DIR, OUTPUT_DIR, MODEL_DIR, TEMP_DIR]:
    os.makedirs(path, exist_ok=True)

display(Markdown("""
## 初期化しています
このまま終わるまでお待ちください。

待っている間に、左の **フォルダのマーク** から **「処理待ち」** フォルダを開いて、
音声ファイルをアップロードしておくと、そのまま次のセルへ進めます。
"""))

def run_quiet(cmd):
    # 通常時の技術ログは抑え、失敗時だけ末尾を表示します。
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stdout[-2000:])

run_quiet(["apt-get", "-qq", "update"])
run_quiet(["apt-get", "-qq", "install", "-y", "ffmpeg"])
run_quiet([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip"])
run_quiet([sys.executable, "-m", "pip", "install", "--quiet", "onnxruntime-gpu>=1.17"])

if not os.path.exists(REPO_DIR):
    run_quiet(["git", "clone", "--no-checkout", REPO_URL, REPO_DIR])

# ノートブック側で内部 API にパッチ当てしているため、上流の変更影響を避ける目的で commit を固定します。
run_quiet(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_COMMIT])
run_quiet(["git", "-C", REPO_DIR, "checkout", "--detach", REPO_COMMIT])

run_quiet([sys.executable, "-m", "pip", "install", "--quiet", "-e", REPO_DIR])

gpu_name = ""
try:
    gpu_result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    gpu_name = gpu_result.stdout.strip().splitlines()[0] if gpu_result.stdout.strip() else ""
except Exception:
    gpu_name = ""

if not gpu_name:
    gpu_note = "### ご確認\n- GPU が見つかりませんでした。Colab の **ランタイムのタイプを変更** から GPU を有効にしてください"
elif "T4" in gpu_name:
    gpu_note = ""
else:
    gpu_note = f"### ご確認\n- GPU として **{gpu_name}** を検出しました。T4 以外でも動くことはありますが、このノートは T4 想定です"

display(Markdown(f"""
## 準備ができました
ありがとうございます。環境の準備が完了しました。

### 次の手順
1. 画面左側の **フォルダのマーク** を押します
2. **「処理待ち」** フォルダを開きます
3. 上の **アップロードボタン** を押すか、音声ファイルをドラッグ＆ドロップしてください
4. アップロードが終わったら、**次のセル** を実行してください

{gpu_note}

### 補足
- 処理前のファイルは **「処理待ち」** に入ります
- 処理後のファイルは **「処理済み」** に保存されます
"""))


In [ ]:
import copy
import json
import logging
import os
import re
import shutil
import subprocess
import sys
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlparse

from IPython.display import Markdown, display

#@markdown ## 2) 処理
#@markdown 下から処理内容を選んで、左の ▶ ボタンを押してください。
#@markdown 2つまで続けて処理できます。迷ったら、まずは **1つだけ** でも大丈夫です。
#@markdown
#@markdown **使い方の目安**
#@markdown - **ノイズが気になるとき** は **big_beta7**
#@markdown - **残響を取りたいとき** は **dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029**
#@markdown - **部屋鳴り感をもう少し狙って抑えたいとき** は **dereverb_room_anvuew_sdr_13.7432**
#@markdown
#@markdown **補足**
#@markdown - **big_beta7** はかなり強力なボーカル分離モデルで、いろいろなノイズに対応しやすいです
#@markdown - **dereverb_room** は **部屋鳴り特化** です。逆に、それ以外のタイプのリバーブにはあまり強くありません
#@markdown - 2つ続けるなら、**リバーブ除去 → ノイズ除去** の順がおすすめです

一つ目の処理 = 'ノイズ除去' #@param ['(なし)', 'ノイズ除去', 'リバーブ除去（自然）', 'リバーブ除去（部屋鳴り向け）']
二つ目の処理 = '(なし)' #@param ['(なし)', 'ノイズ除去', 'リバーブ除去（自然）', 'リバーブ除去（部屋鳴り向け）']

# #@markdown ### 仕上がり
# ていねいに処理する = True #@param {type:"boolean"}
# #@markdown チェックを入れると、少し時間はかかりますが、精度優先で動かします。

#@markdown ### 保存
書き出し形式 = 'FLAC 24bit（おすすめ）' #@param ['FLAC 24bit（おすすめ）', 'FLAC 16bit', 'WAV FLOAT']
# ファイル名をわかりやすくする = True #@param {type:"boolean"}
一段目の結果も残す = True #@param {type:"boolean"}
完成版を自動でダウンロードする = True #@param {type:"boolean"}

first_process = 一つ目の処理
second_process = 二つ目の処理
careful_mode = True # ていねいに処理する
export_format = 書き出し形式
use_modelname = True # ファイル名をわかりやすくする
save_intermediate = 一段目の結果も残す
auto_download = 完成版を自動でダウンロードする

repo_dir = '/content/python-audio-separator'
input_folder = '/content/処理待ち'
output_root = '/content/処理済み'
model_dir = '/content/_pas_models'
temp_root = '/content/_processing_tmp'

model_catalog = {
    'ノイズ除去': {
        'label': 'big_beta7',
        'model_type': 'mel_band_roformer',
        'config_url': 'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta7.yaml',
        'ckpt_url': 'https://huggingface.co/pcunwa/Mel-Band-Roformer-big/resolve/main/big_beta7.ckpt',
        'preferred_output_stem': 'vocals',
        'friendly_suffix': 'ノイズ除去済み',
    },
    'リバーブ除去（自然）': {
        'label': 'dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029',
        'model_type': 'mel_band_roformer',
        'config_url': 'https://huggingface.co/anvuew/dereverb_mel_band_roformer/resolve/main/dereverb_mel_band_roformer_anvuew.yaml',
        'ckpt_url': 'https://huggingface.co/anvuew/dereverb_mel_band_roformer/resolve/main/dereverb_mel_band_roformer_mono_anvuew_sdr_20.4029.ckpt',
        'preferred_output_stem': 'noreverb',
        'friendly_suffix': '自然リバーブ除去済み',
    },
    'リバーブ除去（部屋鳴り向け）': {
        'label': 'dereverb_room_anvuew_sdr_13.7432',
        'model_type': 'bs_roformer',
        'config_url': 'https://huggingface.co/anvuew/dereverb_room/resolve/main/dereverb_room_anvuew.yaml',
        'ckpt_url': 'https://huggingface.co/anvuew/dereverb_room/resolve/main/dereverb_room_anvuew_sdr_13.7432.ckpt',
        'preferred_output_stem': 'noreverb',
        'friendly_suffix': '部屋鳴り除去済み',
    },
}

for spec in model_catalog.values():
    spec['ckpt_filename'] = Path(urlparse(spec['ckpt_url']).path).name
    spec['config_filename'] = Path(urlparse(spec['config_url']).path).name

audio_extensions = {'.wav', '.flac', '.mp3', '.m4a', '.aac', '.ogg', '.opus', '.aif', '.aiff', '.wma'}


class UserFacingError(Exception):
    pass


class StageProcessError(Exception):
    def __init__(self, stage_index, process_name, detail):
        self.stage_index = stage_index
        self.process_name = process_name
        self.detail = detail
        super().__init__(detail)


def show_error(title, body):
    display(Markdown(f'## {title}\n{body}'))


def configure_runtime_messages():
    warnings.filterwarnings('ignore', message='invalid escape sequence', category=SyntaxWarning)
    warnings.filterwarnings('ignore', category=SyntaxWarning, module=r'pydub(\\.|$)')
    logging.getLogger('audio_separator.separator.separator').setLevel(logging.WARNING)
    logging.getLogger('audio_separator.separator.roformer.roformer_loader').setLevel(logging.CRITICAL)


def list_audio_files(folder):
    files = []
    if not os.path.isdir(folder):
        return files
    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)
        if os.path.isfile(path) and Path(name).suffix.lower() in audio_extensions:
            files.append(path)
    return files


def recreate_dir(path):
    if os.path.isdir(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def ensure_ready():
    if not os.path.isdir(repo_dir):
        raise UserFacingError('先に **「1) 準備」セル** を実行してください。')
    if not list_audio_files(input_folder):
        raise UserFacingError('「処理待ち」フォルダに音声ファイルが見つかりませんでした。')


def parse_export_profile(choice):
    if choice == 'FLAC 24bit（おすすめ）':
        return {
            'separator_output_format': 'FLAC',
            'suffix': '.flac',
            'ffmpeg_args': ['-c:a', 'flac', '-sample_fmt', 's32', '-bits_per_raw_sample', '24'],
        }
    if choice == 'FLAC 16bit':
        return {
            'separator_output_format': 'FLAC',
            'suffix': '.flac',
            'ffmpeg_args': ['-c:a', 'flac', '-sample_fmt', 's16'],
        }
    return {
        'separator_output_format': 'WAV',
        'suffix': '.wav',
        'ffmpeg_args': ['-c:a', 'pcm_f32le'],
    }


def download_file(url, destination):
    import requests
    from tqdm.auto import tqdm

    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return destination

    with requests.get(url, stream=True, timeout=300) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))
        with destination.open('wb') as handle, tqdm(total=total, unit='B', unit_scale=True, desc=destination.name) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                handle.write(chunk)
                progress.update(len(chunk))
    return destination


def apply_runtime_patches(runtime_models):
    import torch
    from scipy import signal
    from audio_separator.separator import Separator
    from audio_separator.separator.common_separator import CommonSeparator
    from audio_separator.separator.architectures.mdxc_separator import MDXCSeparator
    from audio_separator.separator.roformer.configuration_normalizer import ConfigurationNormalizer
    from audio_separator.separator.uvr_lib_v5 import spec_utils
    from tqdm.auto import tqdm

    Separator._runtime_custom_models = runtime_models

    if not getattr(Separator, '_runtime_custom_model_patch', False):
        original_download_model_files = Separator.download_model_files

        def patched_download_model_files(self, model_filename):
            runtime_spec = Separator._runtime_custom_models.get(model_filename)
            if runtime_spec is not None:
                return (
                    model_filename,
                    'MDXC',
                    runtime_spec['friendly_name'],
                    str(runtime_spec['model_path']),
                    str(runtime_spec['config_path']),
                )
            return original_download_model_files(self, model_filename)

        Separator.download_model_files = patched_download_model_files
        Separator._runtime_custom_model_patch = True

    def contains_roformer_signature(value):
        if isinstance(value, dict):
            if any(key in value for key in ['freqs_per_bands', 'num_bands', 'mlp_expansion_factor', 'time_transformer_depth', 'freq_transformer_depth']):
                return True
            model_type = value.get('model_type', value.get('architecture', value.get('type')))
            if isinstance(model_type, str) and 'roformer' in model_type.lower():
                return True
            return any(contains_roformer_signature(item) for item in value.values())
        if isinstance(value, (list, tuple)):
            return any(contains_roformer_signature(item) for item in value)
        if isinstance(value, str):
            return 'roformer' in value.lower()
        return False

    CommonSeparator._detect_roformer_model = lambda self: bool(
        self.model_data and (
            self.model_data.get('is_roformer', False)
            or contains_roformer_signature(self.model_data)
            or (self.model_path and 'roformer' in self.model_path.lower())
            or (self.model_name and 'roformer' in self.model_name.lower())
        )
    )

    def patched_detect_model_type(self, config):
        normalized_config = self._normalize_structure(copy.deepcopy(config), '')
        if 'freqs_per_bands' in normalized_config:
            return 'bs_roformer'
        if 'num_bands' in normalized_config or 'n_mels' in normalized_config or 'mel_bands' in normalized_config:
            return 'mel_band_roformer'
        model_type = normalized_config.get('model_type', normalized_config.get('type', normalized_config.get('architecture')))
        if isinstance(model_type, str):
            lowered = model_type.lower()
            if 'bs' in lowered and 'roformer' in lowered:
                return 'bs_roformer'
            if 'mel' in lowered and 'roformer' in lowered:
                return 'mel_band_roformer'
            if 'roformer' in lowered:
                return 'bs_roformer'
        return None

    ConfigurationNormalizer.detect_model_type = patched_detect_model_type

    if not hasattr(MDXCSeparator, '_notebook_original_demix'):
        MDXCSeparator._notebook_original_demix = MDXCSeparator.demix

    def _roformer_uses_stereo_input(self):
        model_stereo = getattr(self.model_run, 'stereo', None)
        if model_stereo is None:
            model_cfg = getattr(self.model_data_cfgdict, 'model', None)
            model_stereo = getattr(model_cfg, 'stereo', None) if model_cfg is not None else None
        return True if model_stereo is None else bool(model_stereo)

    def _run_roformer_overlap_inference(self, mix, chunk_size, step, window, num_stems, device):
        with torch.no_grad():
            req_shape = (num_stems,) + tuple(mix.shape)
            result = torch.zeros(req_shape, dtype=torch.float32)
            counter = torch.zeros(req_shape, dtype=torch.float32)
            for i in tqdm(range(0, mix.shape[1], step)):
                part = mix[:, i:i + chunk_size]
                length = part.shape[-1]
                if i + chunk_size > mix.shape[1]:
                    part = mix[:, -chunk_size:]
                    length = chunk_size
                part = part.to(device)
                x = self.model_run(part.unsqueeze(0))[0].cpu()
                if i + chunk_size > mix.shape[1]:
                    start_idx = result.shape[-1] - chunk_size
                    result = self.overlap_add(result, x, window, start_idx, length)
                    safe_len = min(length, x.shape[-1], window.shape[0])
                    if safe_len > 0:
                        counter[..., start_idx:start_idx + safe_len] += window[:safe_len]
                else:
                    result = self.overlap_add(result, x, window, i, length)
                    safe_len = min(length, x.shape[-1], window.shape[0])
                    if safe_len > 0:
                        counter[..., i:i + safe_len] += window[:safe_len]
        return result / counter.clamp(min=1e-10)

    def patched_demix(self, mix):
        if not self.is_roformer:
            return MDXCSeparator._notebook_original_demix(self, mix)

        orig_mix = mix
        if self.pitch_shift != 0:
            mix, sample_rate = spec_utils.change_pitch_semitones(mix, self.sample_rate, semitone_shift=-self.pitch_shift)

        mix = torch.tensor(mix, dtype=torch.float32)
        mdx_segment_size = self.segment_size if self.override_model_segment_size else self.model_data_cfgdict.inference.dim_t
        num_stems = 1 if self.model_data_cfgdict.training.target_instrument else len(self.model_data_cfgdict.training.instruments)
        stft_hop_len = getattr(self.model_data_cfgdict.model, 'stft_hop_length', None)
        if stft_hop_len is None:
            stft_hop_len = self.model_data_cfgdict.audio.hop_length
        chunk_size = int(stft_hop_len) * (int(mdx_segment_size) - 1)
        desired_step = int(self.overlap * self.model_data_cfgdict.audio.sample_rate)
        step = chunk_size if desired_step <= 0 else min(desired_step, chunk_size)
        window = torch.tensor(signal.windows.hamming(chunk_size), dtype=torch.float32)
        device = next(self.model_run.parameters()).device

        if not self._roformer_uses_stereo_input() and mix.shape[0] > 1:
            inferenced_outputs = torch.cat([
                self._run_roformer_overlap_inference(mix[channel_index:channel_index + 1], chunk_size, step, window, num_stems, device)
                for channel_index in range(mix.shape[0])
            ], dim=1)
        else:
            inferenced_outputs = self._run_roformer_overlap_inference(mix, chunk_size, step, window, num_stems, device)

        if num_stems > 1:
            sources = {}
            for key, value in zip(self.model_data_cfgdict.training.instruments, inferenced_outputs.cpu().detach().numpy()):
                if self.pitch_shift != 0:
                    sources[key] = self.pitch_fix(value, sample_rate, orig_mix)
                else:
                    sources[key] = value
            if self.is_primary_stem_main_target and num_stems == 1:
                if sources[self.primary_stem_name].shape[1] != orig_mix.shape[1]:
                    sources[self.primary_stem_name] = spec_utils.match_array_shapes(sources[self.primary_stem_name], orig_mix)
                sources[self.secondary_stem_name] = orig_mix - sources[self.primary_stem_name]
            return sources

        sources = {k: v.cpu().detach().numpy() for k, v in zip([self.model_data_cfgdict.training.target_instrument], inferenced_outputs)}
        inferenced_output = sources[self.model_data_cfgdict.training.target_instrument]
        if self.pitch_shift != 0:
            primary = self.pitch_fix(inferenced_output, sample_rate, orig_mix)
        else:
            primary = inferenced_output
        if self.is_primary_stem_main_target:
            if primary.shape[1] != orig_mix.shape[1]:
                primary = spec_utils.match_array_shapes(primary, orig_mix)
            secondary = orig_mix - primary
            return {self.primary_stem_name: primary, self.secondary_stem_name: secondary}
        return primary

    MDXCSeparator._roformer_uses_stereo_input = _roformer_uses_stereo_input
    MDXCSeparator._run_roformer_overlap_inference = _run_roformer_overlap_inference
    MDXCSeparator.demix = patched_demix


def build_separator(output_dir, output_profile):
    from audio_separator.separator import Separator
    return Separator(
        model_file_dir=model_dir,
        output_dir=output_dir,
        output_format=output_profile['separator_output_format'],
        log_level=logging.WARNING,
        use_autocast=not careful_mode,
        mdxc_params={
            'segment_size': 256,
            'override_model_segment_size': False,
            'batch_size': 1,
            'overlap': 8,
            'pitch_shift': 0,
        },
    )


def convert_audio_file(input_path, output_profile):
    input_path = Path(input_path)
    temp_path = input_path.with_name(input_path.stem + '__converted' + output_profile['suffix'])
    final_path = input_path.with_suffix(output_profile['suffix'])
    cmd = ['ffmpeg', '-y', '-i', str(input_path), *output_profile['ffmpeg_args'], str(temp_path)]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stdout[-1000:])
    if input_path.exists():
        input_path.unlink()
    if final_path.exists():
        final_path.unlink()
    temp_path.rename(final_path)
    return final_path


def simplify_output_name(path):
    path = Path(path)
    match = re.match(r'^(?P<prefix>.+_\([^)]+\))_.+?(?P<suffix>\.[^.]+)$', path.name)
    if not match:
        return path
    target = path.with_name(match.group('prefix') + match.group('suffix'))
    if target.exists():
        target.unlink()
    path.rename(target)
    return target


def absolute_paths(paths, output_dir):
    resolved = []
    for path in paths:
        candidate = Path(path)
        if not candidate.is_absolute():
            candidate = Path(output_dir) / candidate
        resolved.append(candidate.resolve())
    return resolved


def pick_stage_output(paths, preferred_output_stem):
    token = preferred_output_stem.lower()
    for path in paths:
        lowered = path.name.lower()
        if f'({token})' in lowered or token in lowered:
            return path
    raise FileNotFoundError(f"{preferred_output_stem} の出力が見つかりません: {[p.name for p in paths]}")


def make_session_name(selected_processes):
    labels = [re.sub(r'[（）()]', '', name).replace('向け', '') for name in selected_processes]
    return '＋'.join(labels)


def make_friendly_final_copy(path, friendly_dir, source_stem, selected_processes):
    path = Path(path)
    friendly_dir = Path(friendly_dir)
    friendly_dir.mkdir(parents=True, exist_ok=True)
    suffixes = '・'.join([model_catalog[name]['friendly_suffix'] for name in selected_processes])
    target = friendly_dir / f'{source_stem}_{suffixes}{path.suffix}'
    if target.exists():
        target.unlink()
    shutil.copy2(path, target)
    return target


def prepare_download_target(friendly_outputs, session_root, friendly_output_dir):
    if len(friendly_outputs) == 1:
        return Path(friendly_outputs[0])
    archive_base = session_root / f'{session_root.name}_お渡し用'
    archive_path = Path(str(archive_base) + '.zip')
    if archive_path.exists():
        archive_path.unlink()
    shutil.make_archive(str(archive_base), 'zip', root_dir=str(friendly_output_dir), base_dir='.')
    return archive_path


def trigger_colab_download(path):
    try:
        from google.colab import files as colab_files
    except Exception:
        return False
    colab_files.download(str(path))
    return True


def main():
    ensure_ready()
    selected_processes = [x for x in [first_process, second_process] if x != '(なし)']
    if not selected_processes:
        raise UserFacingError('処理内容が選ばれていません。1つ以上選んでください。')
    if len(selected_processes) != len(set(selected_processes)):
        raise UserFacingError('同じ処理が2つ選ばれています。違う組み合わせにしてください。')

    input_files = list_audio_files(input_folder)
    output_profile = parse_export_profile(export_format)

    configure_runtime_messages()

    runtime_models = {}
    for process_name in selected_processes:
        spec = model_catalog[process_name]
        ckpt_path = download_file(spec['ckpt_url'], Path(model_dir) / spec['ckpt_filename'])
        config_path = download_file(spec['config_url'], Path(model_dir) / spec['config_filename'])
        runtime_models[spec['ckpt_filename']] = {
            'friendly_name': f"Notebook Model: {spec['label']} ({process_name})",
            'model_path': ckpt_path,
            'config_path': config_path,
        }

    sys.path.insert(0, repo_dir)
    apply_runtime_patches(runtime_models)

    jst = timezone(timedelta(hours=9))
    timestamp = datetime.now(jst).strftime('%Y%m%d_%H%M%S')
    session_name = make_session_name(selected_processes)
    session_root = Path(output_root) / f'{timestamp}_{session_name}'
    final_output_dir = session_root / '最終結果'
    friendly_output_dir = session_root / 'お渡し用'
    intermediate_root = session_root / '途中結果'
    final_output_dir.mkdir(parents=True, exist_ok=True)
    friendly_output_dir.mkdir(parents=True, exist_ok=True)
    intermediate_root.mkdir(parents=True, exist_ok=True)
    recreate_dir(temp_root)

    print('処理を始めます。少しお待ちください…')
    print(f'見つかったファイル: {len(input_files)} 件')
    for path in input_files:
        print(' -', Path(path).name)

    stage_summaries = []
    final_outputs = []
    friendly_outputs = []

    for input_path in input_files:
        source_input = Path(input_path)
        current_input = source_input

        for stage_index, process_name in enumerate(selected_processes, start=1):
            spec = model_catalog[process_name]
            is_last_stage = stage_index == len(selected_processes)
            stage_output_dir = final_output_dir if is_last_stage else Path(temp_root) / f'stage_{stage_index}' / current_input.stem
            stage_output_dir.mkdir(parents=True, exist_ok=True)

            print('')
            print(f'--- {stage_index}段目: {process_name} ---')
            print(f"使用モデル: {spec['label']}")

            separator = build_separator(str(stage_output_dir), output_profile)
            separator.load_model(spec['ckpt_filename'])

            try:
                raw_outputs = separator.separate(str(current_input))
            except Exception as exc:
                raise StageProcessError(stage_index, process_name, f'{type(exc).__name__}: {str(exc)[:500]}') from exc

            stage_outputs = [convert_audio_file(path, output_profile) for path in absolute_paths(raw_outputs, stage_output_dir)]
            if not use_modelname:
                stage_outputs = [simplify_output_name(path) for path in stage_outputs]

            carry_output = pick_stage_output(stage_outputs, spec['preferred_output_stem'])
            if save_intermediate and not is_last_stage:
                intermediate_dir = intermediate_root / f'{stage_index}段目_{spec["label"]}' / current_input.stem
                intermediate_dir.parent.mkdir(parents=True, exist_ok=True)
                if intermediate_dir.exists():
                    shutil.rmtree(intermediate_dir)
                shutil.copytree(stage_output_dir, intermediate_dir)

            current_input = carry_output
            stage_summaries.append(f'{stage_index}段目: {process_name}（{spec['label']}）')

        final_outputs.append(current_input)
        friendly_outputs.append(make_friendly_final_copy(current_input, friendly_output_dir, source_input.stem, selected_processes))

    if not save_intermediate and os.path.isdir(temp_root):
        shutil.rmtree(temp_root)

    output_list_md = '\n'.join([f'- `{path.name}`' for path in friendly_outputs]) if friendly_outputs else '- 出力ファイルは見つかりませんでした'
    download_target = prepare_download_target(friendly_outputs, session_root, friendly_output_dir) if friendly_outputs else None
    auto_download_note = ''
    if auto_download and download_target is not None:
        if trigger_colab_download(download_target):
            auto_download_note = f'### ダウンロード\n`{download_target.name}` のダウンロードを開始しました。'
        else:
            auto_download_note = '### ダウンロード\nこの環境では自動ダウンロードを開始できませんでした。必要な場合は左のフォルダから取得してください。'

    display(Markdown(f"""
## 処理が終わりました
おつかれさまでした。保存まで完了しています。

### 今回の処理
{'  \n'.join(stage_summaries)}

### 保存先
左の **フォルダのマーク** から
**「処理済み」 → `{session_root.name}` → `お渡し用`** を開くと、わかりやすい名前の完成版があります。

元のモデル名つきファイルは **`最終結果`** に残しています。

途中の結果を残す設定にした場合は、同じフォルダ内の **`途中結果`** に保存されます。

### 出力ファイル
{output_list_md}

{auto_download_note}
"""))


try:
    main()
except UserFacingError as e:
    show_error('うまく進められませんでした', str(e))
except StageProcessError as e:
    show_error('処理の途中で止まってしまいました', f'**{e.stage_index}段目** の **{e.process_name}** で止まりました。\n\n```text\n{e.detail}\n```')
except Exception as e:
    show_error('予期しないエラーが発生しました', f'`{type(e).__name__}: {str(e)[:500]}`')
